# 01 · Data Acquisition and Scoping

**Project:** *What Makes a Mortgage Cost More?* An interactive app that shows homebuyers
what borrowers like them actually paid for a home-purchase mortgage.

**What this notebook does**

1. Downloads one year of public mortgage data (HMDA) for six states
2. Checks that the download is complete
3. Keeps only the columns the project needs
4. Applies scope filters so every loan is comparable on price
5. Saves a clean starting file for analysis

**Output:** `data/interim/hmda_<YEAR>_purchase_filtered.parquet`, which keeps every application
outcome (approved, denied, withdrawn…) so the same file can be reused in the final project.

> Every data decision in this notebook is numbered (D1, D2, …) and recorded in `DECISIONS.md`.

## Step 1 · Project setup

This cell sets up everything the rest of the notebook needs.

**About the data.** The *Home Mortgage Disclosure Act* (HMDA) requires most US mortgage lenders
to report every application they receive. The public data, published by the CFPB, has one row per
application, with loan details, pricing, outcome, and borrower and neighborhood information.

**Year: 2025 (D2).** One year only. The project question is a snapshot of what borrowers paid,
and pooling several years would mix in the large interest-rate changes between years.

**States: two per home-price tier (D5).** Tiers are based on Zillow's median home value
(July 2025; US median $370,523):

| Tier | Rule | States |
|---|---|---|
| Expensive | Top 5 nationally | California ($809k), Massachusetts ($686k) |
| Mid | Within 20% of the US median | Florida ($405k), Texas ($308k) |
| Low | 25%+ below the US median | Michigan ($260k), Ohio ($246k) |

*Limitation:* both mid-tier states are in the South and both low-tier states are in the Midwest,
so

In [1]:
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt

YEAR = 2025
STATE_TIERS = {"CA": "Expensive", "MA": "Expensive",
               "FL": "Mid",       "TX": "Mid",
               "MI": "Low",       "OH": "Low"}
STATES = list(STATE_TIERS)

# Project folder (also where the downloaded CSV files are)
PROJECT_DIR = Path(r"C:\Users\havil\CMSE_midterm_HMDA_project")
CSV_DIR = PROJECT_DIR

RAW_DIR = PROJECT_DIR / "data" / "raw"          # parquet copies of the CSVs
INTERIM_DIR = PROJECT_DIR / "data" / "interim"  # filtered data
RAW_DIR.mkdir(parents=True, exist_ok=True)
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

print("CSV folder:     ", CSV_DIR)
print("Raw data folder:", RAW_DIR)
print("Interim folder: ", INTERIM_DIR)
print("\nCSV files found:")
for f in sorted(CSV_DIR.glob("*.csv")):
    print(f"   {f.name}  ({f.stat().st_size / 1e6:,.0f} MB)")

CSV folder:      C:\Users\havil\CMSE_midterm_HMDA_project
Raw data folder: C:\Users\havil\CMSE_midterm_HMDA_project\data\raw
Interim folder:  C:\Users\havil\CMSE_midterm_HMDA_project\data\interim

CSV files found:
   loan_purposes_1_state_CA.csv  (177 MB)
   loan_purposes_1_state_FL.csv  (211 MB)
   loan_purposes_1_state_MA.csv  (40 MB)
   loan_purposes_1_state_MI.csv  (69 MB)
   loan_purposes_1_state_OH.csv  (88 MB)
   loan_purposes_1_state_TX.csv  (277 MB)


## Step 2 · Convert the downloaded data to parquet

The data was downloaded manually from the **HMDA Data Browser**
(*Dataset by State* → year 2025 → *Loan Purpose: Home purchase*), one CSV per state.

Here each CSV is converted to **parquet**, a compressed format that loads much faster,
so the large CSVs are read only once.

- **Everything is read as text (`dtype=str`).** HMDA mixes numbers with codes such as `"Exempt"`
  and `"NA"`. If pandas guessed the types, it could silently turn those codes into blanks.
- **`keep_default_na=False`** keeps the literal text `"NA"`, so we can count it ourselves later.
- **Each file's state is detected from its `state_code` column**, so file names don't matter.
- Rerunning this cell skips states that are already converted.

In [5]:
# Convert each downloaded CSV to parquet (done once).
# Each file's state is read from its own state_code column, so CSV file names don't matter.
for csv_file in sorted(CSV_DIR.glob("*.csv")):
    state = pd.read_csv(csv_file, dtype=str, nrows=1)["state_code"].iloc[0]
    if state not in STATES:
        print(f"Skipping {csv_file.name} (state {state} not in project)")
        continue

    out = RAW_DIR / f"hmda_{YEAR}_{state}_purchase.parquet"
    if out.exists():
        print(f"{state}: already converted, skipping")
        continue

    print(f"{state}: converting {csv_file.name}...", end=" ", flush=True)
    state_df = pd.read_csv(csv_file, dtype=str, keep_default_na=False)
    state_df.to_parquet(out, index=False)
    print(f"{len(state_df):,} rows saved")
    del state_df  # free memory before the next file

missing = [s for s in STATES if not (RAW_DIR / f"hmda_{YEAR}_{s}_purchase.parquet").exists()]
print("\n✅ All 6 states ready" if not missing else f"\n⚠️ Missing states: {missing}")

CA: converting loan_purposes_1_state_CA.csv... 465,779 rows saved
FL: already converted, skipping
MA: already converted, skipping
MI: already converted, skipping
OH: already converted, skipping
TX: converting loan_purposes_1_state_TX.csv... 734,539 rows saved

✅ All 6 states ready


In [4]:
for s in ["CA", "TX"]:
    f = RAW_DIR / f"hmda_{YEAR}_{s}_purchase.parquet"
    f.unlink(missing_ok=True)
    print(f"Deleted {f.name}")

Deleted hmda_2025_CA_purchase.parquet
Deleted hmda_2025_TX_purchase.parquet


## Step 3 · Check the data is complete

Before any analysis, we confirm that nothing was lost during download. The total number of rows
must match what the **HMDA Data Browser** reports for the same selection
(6 states, 2025, Loan Purpose: Home purchase): **2,282,462 applications**.

**Why this check matters:** the first downloads of California and Texas were silently cut off.
California had only 57,100 rows instead of 465,779, but the files still opened without any error.
The problem was caught only by comparing row counts with the website, and both states were
re-downloaded.

Row counts are read from the parquet file metadata, so this check is instant and doesn't load the data.

In [6]:
EXPECTED_TOTAL = 2_282_462   # HMDA Data Browser: 6 states, 2025, home purchase
EXPECTED_CA = 465_779        # Data Browser count for California (spot check)

check = pd.DataFrame({
    "state": STATES,
    "tier": [STATE_TIERS[s] for s in STATES],
    "applications": [pq.ParquetFile(RAW_DIR / f"hmda_{YEAR}_{s}_purchase.parquet").metadata.num_rows
                     for s in STATES],
})
check["share_%"] = (check["applications"] / check["applications"].sum() * 100).round(1)
display(check)

total = check["applications"].sum()
ca = check.loc[check["state"] == "CA", "applications"].item()
print(f"Total downloaded: {total:,}  |  Expected: {EXPECTED_TOTAL:,}")
print(f"California:       {ca:,}  |  Expected: {EXPECTED_CA:,}")
print("✅ Data is complete" if (total == EXPECTED_TOTAL and ca == EXPECTED_CA)
      else "⚠️ Mismatch: check which state is too small and re-download it")

,state,tier,applications,share_%
0,CA,Expensive,465779,20.4
1,MA,Expensive,105202,4.6
2,FL,Mid,561246,24.6
3,TX,Mid,734539,32.2
4,MI,Low,182986,8.0
5,OH,Low,232710,10.2


Total downloaded: 2,282,462  |  Expected: 2,282,462
California:       465,779  |  Expected: 465,779
✅ Data is complete


## Step 4 · Load only the columns we need

The raw data has about 99 columns. Loading all of them for 2.28 million rows would use several GB
of memory, so we load about 30, grouped by the role they play in the project:

| Group | Columns | Why |
|---|---|---|
| Identity & location | `activity_year`, `lei`, `state_code`, `county_code`, `census_tract`, `derived_msa-md` | Where the loan is; `lei` identifies the lender (kept for the final project, **D1**) |
| Outcome | `action_taken`, `denial_reason-1` | Approved vs denied; denial reason is for the final project |
| **Pricing** (what we explain) | `interest_rate`, `rate_spread`, `total_loan_costs`, `origination_charges`, `discount_points`, `lender_credits` | The cost of the mortgage |
| Loan features (what drives price) | `loan_type`, `loan_amount`, `loan_term`, `combined_loan_to_value_ratio`, `property_value`, `conforming_loan_limit`, `intro_rate_period`, `interest_only_payment`, `balloon_payment` | Loan size, down payment, loan program, fixed vs adjustable rate |
| Borrower | `income`, `debt_to_income_ratio`, `applicant_credit_score_type`, `derived_race`, `derived_ethnicity`, `derived_sex` | Borrower finances; demographics for later fairness checks |
| Neighborhood | `tract_to_msa_income_percentage`, `tract_minority_population_percent`, `ffiec_msa_md_median_family_income` | Context about the census tract |

We also load 8 **filter-only** columns, used in Step 5 and then dropped.

**Key terms**
- **Rate spread:** how far a loan's rate is above the market average rate for similar loans that week. It measures "how much extra did this borrower pay?"
- **Combined loan-to-value (CLTV):** loan amount ÷ home value. 90% CLTV means a 10% down payment.
- **Intro rate period:** filled in only for adjustable-rate mortgages (ARMs), which are priced differently from fixed-rate loans.
- **Memory:** columns are loaded with `dtype_backend="pyarrow"`, which stores text compactly:
5.58 GB with standard pandas strings vs 0.74 GB with Arrow.

In [9]:
FILTER_COLS = ["loan_purpose", "lien_status", "occupancy_type", "total_units",
               "construction_method", "business_or_commercial_purpose",
               "reverse_mortgage", "open-end_line_of_credit"]

KEEP_COLS = [
    # Identity & location
    "activity_year", "lei", "state_code", "county_code", "census_tract", "derived_msa-md",
    # Outcome
    "action_taken", "denial_reason-1",
    # Pricing
    "interest_rate", "rate_spread", "total_loan_costs", "origination_charges",
    "discount_points", "lender_credits",
    # Loan features
    "loan_type", "loan_amount", "loan_term", "loan_to_value_ratio",
    "property_value", "conforming_loan_limit", "intro_rate_period",
    "interest_only_payment", "balloon_payment",
    # Borrower
    "income", "debt_to_income_ratio", "applicant_credit_score_type",
    "derived_race", "derived_ethnicity", "derived_sex",
    # Neighborhood
    "tract_to_msa_income_percentage", "tract_minority_population_percent",
    "ffiec_msa_md_median_family_income",
]

df = pd.concat(
    [pd.read_parquet(RAW_DIR / f"hmda_{YEAR}_{s}_purchase.parquet",
                     columns=KEEP_COLS + FILTER_COLS,
                     dtype_backend="pyarrow") for s in STATES],
    ignore_index=True,
)
print(f"Loaded {len(df):,} rows × {df.shape[1]} columns")
print(f"Memory used: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB")

Loaded 2,282,462 rows × 40 columns
Memory used: 0.74 GB


## Step 5 · Apply scope filters

To compare mortgage prices fairly, every loan must be the same kind of product. A second mortgage
or an investment-property loan is priced on different terms, so mixing them in would distort the
comparison. These filters define the project's scope (**D3**):

| Filter | Column | Keep code | Why |
|---|---|---|---|
| Home purchase | `loan_purpose` | `1` | Already applied at download; re-checked here |
| First lien | `lien_status` | `1` | Second liens carry more risk and are priced higher |
| Principal residence | `occupancy_type` | `1` | Investor and second-home loans cost more |
| 1–4 units | `total_units` | `1`–`4` | 5+ units are multifamily or commercial loans |
| Site-built home | `construction_method` | `1` | Manufactured homes are financed on different terms |
| Not business purpose | `business_or_commercial_purpose` | `2` or `1111` | Consumer mortgages only |
| Not a reverse mortgage | `reverse_mortgage` | `2` or `1111` | A different product for older homeowners |
| Not an open-end line | `open-end_line_of_credit` | `2` or `1111` | Excludes home-equity lines of credit |

**Why keep code `1111` ("Exempt")? (D4)** Some smaller lenders are legally exempt from reporting
certain fields. Dropping those rows would remove small and community lenders from the data, so we
keep them and flag them in Step 6.

**Why not filter by outcome yet?** We keep approved, denied and withdrawn applications so the
final project (an approval predictor) can reuse this data. The pricing subset is created in Step 8.

The filters are applied one at a time, so the table shows how many rows each one removes.

In [10]:
NO_OR_EXEMPT = ["2", "1111"]  # 2 = No, 1111 = Exempt

steps = [
    ("Home purchase",            df["loan_purpose"] == "1"),
    ("First lien",               df["lien_status"] == "1"),
    ("Principal residence",      df["occupancy_type"] == "1"),
    ("1-4 units",                df["total_units"].isin(["1", "2", "3", "4"])),
    ("Site-built",               df["construction_method"] == "1"),
    ("Not business purpose",     df["business_or_commercial_purpose"].isin(NO_OR_EXEMPT)),
    ("Not reverse mortgage",     df["reverse_mortgage"].isin(NO_OR_EXEMPT)),
    ("Not open-end credit line", df["open-end_line_of_credit"].isin(NO_OR_EXEMPT)),
]

funnel_rows = [("Downloaded", len(df))]
mask = pd.Series(True, index=df.index)
for name, condition in steps:
    mask &= condition
    funnel_rows.append((name, int(mask.sum())))

funnel = pd.DataFrame(funnel_rows, columns=["step", "rows_remaining"])
funnel["removed"] = (funnel["rows_remaining"].shift(1) - funnel["rows_remaining"]).fillna(0).astype(int)
funnel["pct_of_downloaded"] = (funnel["rows_remaining"] / len(df) * 100).round(1)
display(funnel)

,step,rows_remaining,removed,pct_of_downloaded
0,Downloaded,2282462,0,100.0
1,Home purchase,2282462,0,100.0
2,First lien,2200715,81747,96.4
3,Principal residence,1940913,259802,85.0
4,1-4 units,1940898,15,85.0
5,Site-built,1760505,180393,77.1
6,Not business purpose,1757095,3410,77.0
7,Not reverse mortgage,1756317,778,76.9
8,Not open-end credit line,1747609,8708,76.6


### Reading the filter funnel

The scope filters keep **1,747,609 of 2,282,462 applications (76.6%)**.

- **Principal residence** removes the most rows (259,802; 11.8%): investor and second-home
  purchases, which are priced higher and would distort comparisons.
- **Site-built** removes 180,393 (9.3%): manufactured homes, which are financed on different terms.
- **First lien** removes 81,747 (3.6%): second mortgages taken at purchase, such as
  "piggyback" loans that cover part of the down payment.
- **Home purchase** removes nothing, confirming the download filter worked.
- The remaining filters remove very few rows (1–4 units: 15; business, reverse and open-end
  loans: 12,896 combined).

## Step 6 · Flag Exempt rows and save

We add a `scope_exempt_flag` column that is `True` when a row passed a filter only because the field
was `Exempt` (code `1111`). This lets us later check whether results change when these rows are
excluded (a *sensitivity check*).

The filter-only columns are then dropped, since every remaining row has the same values for them,
and the result is saved. This file keeps **all application outcomes** and is the starting point for
both the midterm and the final project.

In [11]:
scoped = df[mask].copy()
scoped["scope_exempt_flag"] = (
    scoped[["business_or_commercial_purpose", "reverse_mortgage", "open-end_line_of_credit"]]
    .eq("1111").any(axis=1)
)
scoped = scoped.drop(columns=FILTER_COLS).reset_index(drop=True)

out_path = INTERIM_DIR / f"hmda_{YEAR}_purchase_filtered.parquet"
scoped.to_parquet(out_path, index=False)

print(f"Saved {len(scoped):,} rows × {scoped.shape[1]} columns → {out_path}")
print(f"File size: {out_path.stat().st_size / 1e6:.1f} MB")
print(f"Rows flagged as Exempt in a scope field: {scoped['scope_exempt_flag'].sum():,} "
      f"({scoped['scope_exempt_flag'].mean():.2%})")

del df  # the full table is no longer needed; frees memory

Saved 1,747,609 rows × 33 columns → C:\Users\havil\CMSE_midterm_HMDA_project\data\interim\hmda_2025_purchase_filtered.parquet
File size: 55.4 MB
Rows flagged as Exempt in a scope field: 19,965 (1.14%)


## Step 7 · What happened to each application?

`action_taken` records each application's outcome:

| Code | Meaning |
|---|---|
| 1 | Loan originated (approved and taken) |
| 2 | Approved but not accepted by the applicant |
| 3 | Denied |
| 4 | Withdrawn by the applicant |
| 5 | File closed as incomplete |
| 6 | Purchased loan (bought from another lender, not a new application) |
| 7 / 8 | Preapproval request denied / approved but not accepted |

The table shows the share (%) of each outcome within each state.

ACTION_LABELS = {
    "1": "Originated", "2": "Approved, not accepted", "3": "Denied",
    "4": "Withdrawn", "5": "Incomplete", "6": "Purchased loan",
    "7": "Preapproval denied", "8": "Preapproval approved, not accepted",
}
scoped["outcome"] = scoped["action_taken"].astype(str).map(ACTION_LABELS)

outcomes = pd.crosstab(scoped["state_code"], scoped["outcome"], normalize="index").mul(100).round(1)
outcomes = outcomes.loc[STATES]  # keep tier order
outcomes.insert(0, "tier", [STATE_TIERS[s] for s in STATES])
outcomes.insert(1, "applications", scoped["state_code"].value_counts().loc[STATES].values)
display(outcomes)

In [12]:
ACTION_LABELS = {
    "1": "Originated", "2": "Approved, not accepted", "3": "Denied",
    "4": "Withdrawn", "5": "Incomplete", "6": "Purchased loan",
    "7": "Preapproval denied", "8": "Preapproval approved, not accepted",
}
scoped["outcome"] = scoped["action_taken"].astype(str).map(ACTION_LABELS)

outcomes = pd.crosstab(scoped["state_code"], scoped["outcome"], normalize="index").mul(100).round(1)
outcomes = outcomes.loc[STATES]  # keep tier order
outcomes.insert(0, "tier", [STATE_TIERS[s] for s in STATES])
outcomes.insert(1, "applications", scoped["state_code"].value_counts().loc[STATES].values)
display(outcomes)

outcome,tier,applications,"Approved, not accepted",Denied,Incomplete,Originated,"Preapproval approved, not accepted",Preapproval denied,Purchased loan,Withdrawn
state_code,,,,,,,,,,
CA,Expensive,368241,2.0,5.6,1.4,55.9,1.7,0.2,19.1,14.1
MA,Expensive,83711,1.1,4.1,1.6,59.4,2.0,0.3,19.8,11.6
FL,Mid,418606,1.4,8.1,1.4,53.9,1.0,0.2,21.0,13.0
TX,Mid,553096,1.7,6.2,1.4,52.9,1.1,0.2,23.6,12.8
MI,Low,136474,1.4,5.5,1.7,63.5,2.7,0.3,12.3,12.5
OH,Low,187481,1.4,4.6,1.4,58.6,1.6,0.3,18.8,13.3


## Step 8 · Pricing subset: originated loans only

The midterm question is about **how much a mortgage costs**, and a price exists only when a loan
was actually made. For denied or withdrawn applications, the pricing fields are blank **by design**:
no loan was made, so there is no rate to report.

That's why we **don't impute** the missing prices (**D7**). Filling them in would invent prices for
loans that never existed. Instead, the pricing analysis uses only originated loans
(`action_taken = 1`).

In [13]:
priced = scoped[scoped["action_taken"] == "1"]

by_state = (priced["state_code"].value_counts().loc[STATES]
            .rename("originated_loans").to_frame())
by_state.insert(0, "tier", [STATE_TIERS[s] for s in STATES])
by_state["share_of_applications_%"] = (
    by_state["originated_loans"] / scoped["state_code"].value_counts().loc[STATES] * 100
).round(1)
display(by_state)
print(f"Total originated loans for the pricing analysis: {len(priced):,}")

,tier,originated_loans,share_of_applications_%
state_code,,,
CA,Expensive,205767,55.9
MA,Expensive,49761,59.4
FL,Mid,225732,53.9
TX,Mid,292354,52.9
MI,Low,86675,63.5
OH,Low,109912,58.6


Total originated loans for the pricing analysis: 970,201
